In [1]:
# =========================
# IMPORTS
# =========================
import csv          #Imports data from csv file
import json         #Allows saving results to file
import logging      #Logging setup
import math         # needed for sqrt
import datetime     #To timestamp output files

# =========================
# THRESHOLDS
# =========================

THRESHOLDS = {        #Sets thresholds that warn if measurements exceed 
    'nitrate': 3,     #regulatory or ecological thresholds.
    'temp': 20        #ALL CAPS = Python convention for constants
}

# =========================
# LOGGING SETUP
# =========================

logging.basicConfig(     #sets op abailty to log errors
    level=logging.INFO,  # minimum level to record
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='analysis.log',  # logs will be written to this file
    filemode='a'  # append the entries to the file or 'w' to overwrite each run
)

# =========================
# LOAD CSV FUNCTION
# =========================
def load_csv(filename):
    """
    Load a CSV file and return a list of dictionaries.
    Converts numeric fields to int/float automatically.
    """
    data = []

    try:
        with open(filename, "r") as file:    #Standard loading loop
            reader = csv.DictReader(file)
            
            for row in reader:
                
                # Convert to numeric values
                try:
                    row['day'] = int(row['day'])
                except ValueError:     #If data is missing or corrupt in day, returns None
                    row['day'] = None

                try:
                    row['temp'] = float(row['temp'])
                except ValueError:      #If data is missing or corrupt in temp, returns None
                    row['temp'] = None  

                try:
                    row['nitrate'] = float(row['nitrate'])
                except ValueError:       #If data is missing or corrupt in nitrate, returns None
                    row['nitrate'] = None

                data.append(row)

    except FileNotFoundError:     #Logs an error if file can't be opened
        logging.error(f"File '{filename}' not found.")
        return []

    if not data:                  #Logs an error if dataset is empty
        logging.warning(f"File '{filename}' was empty or all rows invalid.")

    return data     #Stores the loaded and converted data as 'data'
   

# =========================
# FILTER FUNCTION
# =========================
def filter_by_field(data, field_name, value):
    """Filters the dataset by field name (eg 'group' or 'day')"""
    filtered = []                          #creates an empty list to store the results
    for record in data:                    #loops through each record and appends the whole record to the list
        if record[field_name] == value:    #if record[field_name] equals the group we are currently filtering for
            filtered.append(record)        #... ie the site or day name from get_unique_values  ie site 'A' == 'A'
    return filtered                        #..then append to and return the list

# =========================
# COMPUTE MAX & MIN VALUES FUNCTION
# =========================

def compute_max(data, variable):
    """Calculates the maxiumum temp or nitrate filtered by site or day."""
    if len(data) == 0:     #If the list is empty retrun None
        return None

    #Set min_record to None
    max_record = None       

    #Loops through each record in the data set 
    #and sets the value of the relevant variable to a variable called value
    for record in data:
        value = record[variable]

        #If value is None(record in dataset is corrupt or missing) skip this record
        if value is None:
            continue

        #If the data is OK and the value is less than  
        #the previous min_record value it replaces that value.
        #It also replaces the previous value if that value were still None
        if max_record is None or value > max_record[variable]:
            max_record = record

    return max_record


def compute_min(data, variable):
    """Calculates the maxiumum temp or nitrate filtered by site or day."""
    #Protects against empty dataset
    if len(data) == 0:
        return None

    #Set min_record to None
    min_record = None

    #Loops through each record in the data set 
    #and sets the value of the relevant variable to a variable called value
    for record in data:
        value = record[variable]

        #If value is None(record in dataset is corrupt or missing) skip this record
        if value is None:
            continue
       
        #If the data is OK and the value is less than  
        #the previous min_record value it replaces that value.
        #It also replaces the previous value if that value were still None
        if min_record is None or value < min_record[variable]:
            min_record = record

    return min_record
    
# =========================
# COMPUTE AVERAGE FUNCTION
# =========================

def compute_average(dataset, variable):
    """Calculates the average temp or nitrate filtered by site or day."""
    total = 0
    count = 0 #Count = number of valid entries in the list we are working on
    
    for record in dataset:
        value = record[variable]  # get the value from the record
        if value is not None:
            total += value    #Adds current value to the others already assessed
            count += 1        #Adds 1 to the number of records assessed
            
    if count == 0:
        logging.warning(f"No valid data found for '{variable}' when computing average.")
        return None  # if no valid data to average, with warning logged in line above

    average = total / count    #Typical average calculation

    return average
    
# ==========================
# COMPUTE STANDARD DEVIATION
# ==========================

def compute_std(dataset, variable):
    """
    Computes standard deviation for a variable in a dataset.
    Ignores None values.
    """
    values = []
    for record in dataset:
        value = record[variable]       #Loops through each record in the dataset.
        if value is not None:          #Checks if the value of record[variable] is not None.
            values.append(value)       #If it isn’t None, it adds it to a new list called values.
    
    n = len(values)
    if n < 2:
        logging.warning(f"Not enough valid data to compute standard deviation for '{variable}'.")
        return None
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / (n - 1)  # sample SD
    return math.sqrt(variance)

# =============
# COMPUTE TREND
# =============
    
def compute_trend(dataset, variable):
    """
    Determines whether a variable shows an increasing, decreasing,
    or stable trend across the dataset.
    """

    values = []         #Create a list of stoe values

    for record in dataset:  #Loop through the records

        if record[variable] is not None:   
            values.append(record[variable])

    if len(values) < 2:    #Return None if less than 2 records (too few to see a trend)
        return None

    change = values[-1] - values[0]    #Compare last value [-1] with first value [0]

    if change > 0.5:
        return "Increasing"

    elif change < -0.5:
        return "Decreasing"

    else:
        return "Stable"    

# ===============
#DETECT ANOMALIES
# ===============

def detect_anomalies(dataset, variable):
    """
    Detects unusual values in the dataset using a
    2-standard-deviation rule.
    Returns a list of records containing anomalies.
    """

    avg = compute_average(dataset, variable)
    std = compute_std(dataset, variable)

    if avg is None or std is None:
        return []

    anomalies = []    #Creates a list to store values

    for record in dataset:   #Loops through each record

        value = record[variable]    

        if value is not None:

            if abs(value - avg) > 2 * std:   #(value − mean) > 2 × standard deviation
                                             #A quick rule used in exploratory analysis
                anomalies.append(record)     #Adds detected anomalies to the list

    return anomalies
    
# =========================
# ANALYSIS FUNCTION
# =========================
def analyze_variable(data, variable):
    """Calls all the calculation functions."""

    if len(data) == 0:    #Returns None if there is no data
        return None

    return {
        "average": compute_average(data, variable),
        "max": compute_max(data, variable),
        "min": compute_min(data, variable),
        "std": compute_std(data, variable),
        'trend': compute_trend(data, variable),
        'warning': None,    #Warning of exceeded threshold, set to None by default
        'anomalies': detect_anomalies(data, variable)
    }

# =========================
# GET UNIQUE VALUES FUNCTION
# =========================
def get_unique_values(data, field_name): 
    """Gives you the group keys (like 'Site A' or 1 (day)"""
    values = set()                        #Creates an empty set. A set automatically stores only unique values (no duplicates)
    for record in data:                   #This structures the master_results dictionary so the program knows...
        values.add(record[field_name])    #...these are all the groups I need to analyse '(site' or 'day'), and then loops over them once.
    return values

# =========================
# BUILD MASTER RESULTS FUNCTION
# =========================
def build_master_results(data, group_field, variables):
    """Get all unique values for the grouping field (e.g. all sites or all days)"""
    groups = get_unique_values(data, group_field)
    
    master_results = {}     # Create the main results dictionary that will store the full analysis
                            # Structure will eventually look like:
                            # { group_value : { variable : {avg, max, min} } } master_results = {}

    # Loop through each group (e.g. Site A, Site B, Site C)
    for group in groups:  

        #Logs group level activity
        logging.info(f"Analyzing group: {group}")
        
        # Filter the dataset to include only records belonging to this group
        # Example: all rows where site == "A"
        group_data = filter_by_field(data, group_field, group)
        
        # Create an empty dictionary inside master_results for this group
        master_results[group] = {}

        # Loop through each variable we want to analyze (e.g. temp, nitrate)
        for var in variables:
            
            #Logs variable level activity
            logging.info(f"  Analyzing variable: {var}")

            # Run the statistical analysis for this variable on the filtered data
            # analyze_variable() returns a dictionary containing:
            # { 'average': value, 'max': record, 'min': record }

            #Temporarily store the results in a variable (stats)...
            #...before inserting them into the dictionary.
            stats = analyze_variable(group_data, var)  
            master_results[group][var] = stats 

            if var in THRESHOLDS:                  #Is there a threshold for this variable?
                
                threshold = THRESHOLDS[var]      #variable with threshold value to be compared to

               # Collect all records exceeding the threshold
                exceedances = []

                for record in group_data:   #Loop through every record in group_data

                    if record[var] is not None and record[var] > threshold:  #Check value is not None and if it exceeds threshold

                        exceedances.append(record)   #If it does, append the entire record to the list
                    
                if exceedances:
                    # Build a warning message listing all days

                    messages = []                #Create a list to to store records

                    for record in exceedances:   #Loops through all the records that exceeded the threshold

                        text = f"{var}={record[var]} (day {record['day']})"  #Build a readable message with the variable, its value and the relevant day

                        messages.append(text)     

                    warning_msg = "; ".join(messages)      #Joins more than one warning message together if needed
                    stats['warning'] = f"Threshold exceeded: {warning_msg}"    #Stores warning message in stats['warning']
                    logging.warning(f"{group} {stats['warning']}")    #Logs a warning if exceeded in the stats dictionary

            
            if stats['anomalies']:    #Check for anomalies

                messages = []                    #Create a list to to store records

                for record in stats['anomalies']:   #Loops through all the records that have an anomaly

                    messages.append(
                        f"{var}={record[var]} (day {record['day']})"
                    )

                anomaly_msg = "; ".join(messages)       #Joins more than one warning message together if needed

                logging.warning(
                    f"{group} anomaly detected: {anomaly_msg}"   #Logs a warning
                )
            
            if stats['std'] is not None and stats['average'] != 0:  #Warning for too high standard deviation
                if stats['std'] > 0.5 * stats['average']:           #High being more than 50%, logs a warning
                    logging.warning(f"High variability for {group} {var}: std={stats['std']:.2f}")
            
            if stats['trend'] is not None:
                logging.info(f"{group} {var} trend: {stats['trend']}")
                        
            # Optional warnings (if group has no data or partially missing records)
            if stats['average'] is None:
                logging.warning(f"No valid data to compute average for {group} - {var}")
            if stats['max'] is None or stats['min'] is None:
                logging.warning(f"Some records missing for {group} - {var}")
            if stats['std'] is None:
                logging.warning(f"Standard deviation could not be computed for {group} - {var}")
                   
            std_value = stats['std']    #Stops a crash if std = None
            std_text = f"{std_value:.2f}" if std_value is not None else "None"
        
            #Logs the results of the analysis (values) as a full record
            logging.info(
                f"{group} - {var} | "
                f"avg={stats['average']:.2f} | "     #:.2f gives answer with 2 decimal places
                f"max={stats['max']} | "
                f"min={stats['min']} | "
                f"std={std_text}"
            )

    # Run summary, logging
    logging.info("Finished building master results")
    logging.info(f"Groups analyzed: {len(master_results)}") #Number of sites or days analysed
    logging.info(f"Variables analyzed per group: {len(variables)}")  #Number of variables (temp, nitrate) analysed

    total_stats = len(master_results) * len(variables)
    logging.info(f"Total variable analyses completed: {total_stats}")

    # After all groups and variables have been processed,
    # return the completed nested results dictionary
    return master_results

        
# =========================
# CLASS DEFINITION
# =========================

class RiverAnalysis:
    """An environmental model of a river"""
    
    def __init__(self, data):
        #Initialise name and age attributes
        self.data = data
        self.master_results = None

    def compute_average(self, variable):
        #Call the compute_average function from within the class
        return compute_average(self.data, variable)
        
    # =========================
    # MASTER CONTROL FUNCTION
    # =========================
    
    def run_analysis(self, group_field, variables):
        """Runs an analysis of the uploaded dataset."""

        logging.info("======================================")
        logging.info("Starting River Analysis Pipeline")
        logging.info("======================================")
        
        #Metadata logging
        logging.info("===== ANALYSIS RUN STARTED =====")
        logging.info(f"Dataset size: {len(self.data)} records")
        logging.info(f"Grouping field: {group_field}")
        logging.info(f"Variables selected: {variables}")
     
        # Validate group field (checking field name is in the first record)
        if group_field not in self.data[0]:
            print(f"Error: Group field '{group_field}' not found in dataset.")
            return False    #Stops the function, execution jumps back 
                            #to where the function was called
            
        # Validate variables (checking variable name is a key in first record)
        for var in variables:
            if var not in self.data[0]:
                print(f"Error: Variable '{var}' not found in dataset.")
                return False      #Stops the function

        #Confirm dataset passed validation.
        logging.info("Input validation successful")
        # Store settings in instance
        self.group_field = group_field
        self.variables = variables

        # Run analysis
        self.master_results = build_master_results(self.data, group_field, variables)

        #Log completion of analysis
        logging.info("=================================")
        logging.info("Analysis pipeline completed")
        logging.info("===== ANALYSIS RUN FINISHED =====")
        
        return True 

    # =========================
    # PRINT REPORT FUNCTION
    # =========================
    def print_report(self):
        """
        Prints group-level statistics (from self.master_results)
        and overall averages for self.data.
        """
        
        logging.info("Generating printed analysis report")  #Helps trace program flow
        
        if self.master_results is None:  #If master_results is empty, print error message
            print("No results available. Run analysis first.")
            return

        # Step 2a: Overall averages
        if self.data is not None: #If master_results is not empty
            print("\n=== Overall Averages =====")
            for var in self.variables:      #For each variable in turn
                overall_avg = compute_average(self.data, var)
                overall_std = compute_std(self.data, var)
                print(f"{var.title()}: Avg={overall_avg:.2f} std={overall_std:.2f}")   #Print variable name & overall average
        print("==========================\n")
    
        # Step 2b: Group-level averages
        for group in sorted(self.master_results):    #For each group (site or day)
            print("\n--------------------------------")
            print(f"Group: {group}")
            print("--------------------------------")
            
            for var in self.variables:   #For each variable within that selected group
                
                stats = self.master_results[group][var]   #Set variable stats to catch the value of the variable within the group

                max_record = stats['max']   #Stores whatever stats[max] contained
                min_record = stats['min']   #Stores whatever stats[max] contained
                avg_value = stats['average'] #Stores whatever stats[average] contained
                std_value = stats['std']
                
                print(f"  {var}")           #Print the variable name
                
                if max_record is not None:   #If there is valid data in max (ie compute_max did not return None)
                    print(f"    Max:     {stats['max'][var]} (day {stats['max']['day']})")   #Print max value on relevant day
                else:            #Otherwise print an error message
                    print("    Max: No valid data")

                if min_record is not None:   #If there is valid data in min (ie compute_min did not return None)
                    print(f"    Min:     {stats['min'][var]} (day {stats['min']['day']})")   #Print min value on relevant day
                else:            #Otherwise print an error message
                    print("    Min: No valid data")

                if avg_value is not None:    #If there is valid data in average (ie compute_average did not return None)
                    print(f"    Avg:     {stats['average']:.2f}")     #Print average for that variable at that site
                else:            #Otherwise print an error message
                    print("    Avg: No valid data")

                if std_value is not None:     #If there is valid data in standard deviation (ie compute_std did not return None)
                    print(f"    Std Dev: {stats['std']:.2f}")         #Print standard deviation for that variable at that site
                else:            #Otherwise print an error message
                    print("    Std: Not enough data")

                if stats['trend'] is not None: #If there is valid data in trend (ie trend did not return None)
                    print(f"    Trend:   {stats['trend']}")          #Print trend for that variable at that site
                else:            #Otherwise print an error message
                    print("    Trend: Not enough data")

                if stats['warning'] is not None:   #Prints warning message if thresholds exceeded
                    print(f"    WARNING: {stats['warning']}")

                if stats['anomalies']:  #Needs a loop as anomalies is a list of dictionaries

                    messages = []

                    for record in stats['anomalies']:

                        messages.append(
                            f"{var}={record[var]} (day {record['day']})"   #Collects abnormal varable & relevant day first 
                        )

                    anomaly_msg = "; ".join(messages)    #Joins all anomalies together in a list

                    print(f"    ANOMALY: {anomaly_msg}")
    
    # =========================
    # SAVE RESULTS TO FILE FUNCTION
    # =========================
    def save_results_to_file(self, filename):
        """Saves master_results to a file."""
        with open(filename, "w") as file:   #'w' overwrites the previous file contents
            json.dump(self.master_results, file)
        print(f"\nResults saved to {filename}")   #Prints confirmation


    # =================================
    # SAVE RESULTS TO CSV FILE FUNCTION
    # =================================   

    def save_results_to_csv(self, filename):
        """Save analysis results to a CSV file."""
    
        with open(filename, "w", newline="") as file:
            writer = csv.writer(file)     #Creates a CSV writing tool
            
            # Write header - Defines the column names (all the names of data in master_results)
            writer.writerow([            #Basically you are just writing the first (top) row
                "group",
                "variable",
                "average",
                "std",
                "max_value",
                "max_day",
                "min_value",
                "min_day"
            ])

            # Write rows
            for group in self.master_results:   #Loops through groups (A, B, C etc)
                for var in self.variables:      #Loops through variables (temp, nitrate)

                    stats = self.master_results[group][var]    #Retrieves the master_results dictionary and stroes the data as a variable

                    max_record = stats['max']    #Allows us to extract values from within the max and min lists
                    min_record = stats['min']

                    writer.writerow([            #Writes all the other rows, one after the other, top down, in the loop
                        group,
                        var,
                        stats['average'],
                        stats['std'],
                        max_record[var] if max_record else None,     #Pulls the values from max-record/min_record
                        max_record['day'] if max_record else None,   #as long as there is no missing data
                        min_record[var] if min_record else None,     #Step 1 — retrieve the max/min record (max_record = stats['max'])
                        min_record['day'] if min_record else None    #Step 2 — retrieve the value and day from that record (max_reord[var])
                    ])

        print(f"\nCSV results saved to {filename}")

    # ==============================
    # SAVES RESULTS TO JSON AND CSV 
    # ===============================

    def save_results(self, timestamp):
        """Save results to both JSON and CSV using timestamped filenames."""

        #Create variables to hold the time formatted complete filename
        json_filename = f"analysis_results_{timestamp}.json"
        csv_filename = f"analysis_results_{timestamp}.csv"

        #Save results to JSON file
        self.save_results_to_file(json_filename)
        #Save results to CSV file
        self.save_results_to_csv(csv_filename)

        #Log the successful creation of files
        logging.info(f"Results written to {json_filename} and {csv_filename}")


# =========================
# CONTROL PANEL
# =========================

# Specify group field and variables to analyze
group_field = 'site'          # Could also be 'day'
variables = ['temp', 'nitrate']

# Load the dataset from CSV
DATA_FILE = "river_data.csv"       #Easier to change dataset
river_data = load_csv(DATA_FILE)

if not river_data:  
    logging.warning("No data loaded. Program stopped.")
else:
    # Create an instance
    analysis = RiverAnalysis(river_data)

    #Python runs run_analysis, if analysis ran successfully it returns True
    success = analysis.run_analysis(group_field, variables)

    #If success is True, the save file runs
    if success:
        # Print report
        analysis.print_report()

        #Create a variable to hold the current date and time (datetime) converted to a formatted string (strftime)
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

        #Save reults to json and csv files
        analysis.save_results(timestamp)
        
        #Log the successful analysis
        logging.info("Analysis completed successfully.")

        # Load results back from file (optional check)
        with open("analysis_results.json", "r") as file:
            loaded_results = json.load(file)



=== Overall Averages =====
Temp: Avg=16.26 std=1.36
Nitrate: Avg=2.77 std=0.43


--------------------------------
Group: A
--------------------------------
  temp
    Max:     14.9 (day 2)
    Min:     14.2 (day 1)
    Avg:     14.55
    Std Dev: 0.49
    Trend:   Increasing
  nitrate
    Max:     2.3 (day 2)
    Min:     2.1 (day 1)
    Avg:     2.20
    Std Dev: 0.14
    Trend:   Stable

--------------------------------
Group: B
--------------------------------
  temp
    Max:     17.2 (day 2)
    Min:     16.8 (day 1)
    Avg:     17.00
    Std Dev: 0.28
    Trend:   Stable
  nitrate
    Max:     3.4 (day 1)
    Min:     3.1 (day 2)
    Avg:     3.25
    Std Dev: 0.21
    Trend:   Stable

--------------------------------
Group: C
--------------------------------
  temp
    Max:     15.9 (day 2)
    Min:     15.5 (day 1)
    Avg:     15.70
    Std Dev: 0.28
    Trend:   Stable
  nitrate
    Max:     2.8 (day 1)
    Min:     2.6 (day 2)
    Avg:     2.70
    Std Dev: 0.14
    Trend: 